# Projeto Avaliativo - Módulo 1  
## Pipeline para Manutenção Preditiva na Indústria 4.0

Este notebook segue a estrutura do **Projeto Avaliativo - Módulo 1** e utiliza a base local `data/manutencao_preditiva.csv`.

**Objetivo:** construir um pipeline preditivo capaz de classificar se uma máquina tende a apresentar falha mecânica (`falha_maquina = 1`) ou operar normalmente (`falha_maquina = 0`).

**Atenção contra vazamento de dados:**  
A coluna `falha_maquina` é usada exclusivamente como variável alvo (`y`). Ela não entra em `X`.  
As colunas `falha_twf`, `falha_hdf`, `falha_pwf`, `falha_osf` e `falha_rnf` também são removidas das variáveis preditoras porque representam motivos técnicos da falha já ocorrida, e não sensores independentes para previsão.

## 0. Importação das bibliotecas e carregamento local da base

In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

# Leitura local: sem Kaggle e sem caminho absoluto.
DATA_PATH = Path("data") / "manutencao_preditiva.csv"

df = pd.read_csv(DATA_PATH)
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\manutencao_preditiva.csv'

## Fase 1: Análise Exploratória dos Dados (EDA)

Nesta etapa são avaliadas as dimensões do dataset, tipos de variáveis, estatísticas descritivas e padrões visuais relevantes para a modelagem.

In [ ]:
print(f"Dimensões do dataset: {df.shape[0]} linhas e {df.shape[1]} colunas")
display(df.info())
display(df.describe())

In [ ]:
# Gráfico 1: distribuição da variável alvo
contagem_alvo = df["falha_maquina"].value_counts().sort_index()
percentuais = 100 * contagem_alvo / len(df)

plt.figure(figsize=(6, 4))
barras = plt.bar(contagem_alvo.index.astype(str), contagem_alvo.values)
plt.title("Distribuição da variável alvo: falha_maquina")
plt.xlabel("Falha da máquina")
plt.ylabel("Quantidade de registros")

for i, valor in enumerate(contagem_alvo.values):
    plt.text(i, valor, f"{percentuais.iloc[i]:.2f}%", ha="center", va="bottom")

plt.show()

In [ ]:
# Gráfico 2: histogramas das principais variáveis preditoras numéricas
variaveis_numericas_eda = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min"
]

df[variaveis_numericas_eda].hist(figsize=(12, 8), bins=30)
plt.suptitle("Distribuição das variáveis preditoras numéricas", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Gráfico 3: correlação de Pearson entre variáveis numéricas
plt.figure(figsize=(10, 7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Mapa de calor - Correlação de Pearson")
plt.show()

### Análise da EDA

A base apresenta forte desbalanceamento da variável alvo: a maior parte dos registros representa funcionamento normal, enquanto as falhas são minoria. Esse padrão exige balanceamento apenas nos dados de treino para evitar que os modelos aprendam a priorizar a classe majoritária.

As variáveis contínuas possuem escalas diferentes, principalmente `velocidade_rotacao_rpm`, `torque_nm` e `desgaste_ferramenta_min`. Por isso, o KNN precisará de escalonamento com `StandardScaler`. A Árvore de Decisão não depende de distância entre pontos e, portanto, será mantida sem escalonamento.

O mapa de correlação ajuda a identificar relações lineares entre variáveis e possíveis redundâncias, mas a seleção final também considera o risco de vazamento de dados.

## Fase 2: Limpeza e Tratamento de Dados (Data Prep)

In [ ]:
df_tratado = df.copy()

duplicadas = df_tratado.duplicated().sum()
print(f"Linhas duplicadas encontradas: {duplicadas}")

df_tratado = df_tratado.drop_duplicates()
print(f"Dimensões após remoção de duplicadas: {df_tratado.shape}")

In [ ]:
nulos = df_tratado.isna().sum().sort_values(ascending=False)
display(nulos[nulos > 0])

In [ ]:
# Avaliação simples de assimetria para justificar média ou mediana
colunas_com_nulos = df_tratado.columns[df_tratado.isna().any()].tolist()
assimetria = df_tratado[colunas_com_nulos].skew(numeric_only=True).sort_values(ascending=False)
display(assimetria)

### Justificativa da imputação

Foi adotada a **mediana** para imputação dos valores ausentes nas variáveis numéricas. Essa escolha é mais robusta quando há assimetria ou presença de outliers, pois a mediana é menos influenciada por valores extremos do que a média.

In [ ]:
colunas_numericas = df_tratado.select_dtypes(include=np.number).columns.tolist()

imputador_mediana = SimpleImputer(strategy="median")
df_tratado[colunas_numericas] = imputador_mediana.fit_transform(df_tratado[colunas_numericas])

print("Valores ausentes após imputação:")
display(df_tratado.isna().sum().sort_values(ascending=False).head(10))

In [ ]:
# Boxplots para identificar outliers nas variáveis explicativas
variaveis_explicativas_numericas = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min"
]

plt.figure(figsize=(12, 6))
df_tratado[variaveis_explicativas_numericas].boxplot(rot=45)
plt.title("Boxplots das variáveis explicativas numéricas")
plt.ylabel("Valores")
plt.tight_layout()
plt.show()

### Análise dos outliers

Os boxplots permitem observar valores extremos nas variáveis operacionais. Como esses registros podem representar condições reais de operação industrial próximas de falhas, eles não foram removidos automaticamente. A decisão foi preservar a informação e usar modelos capazes de lidar com esses padrões.

## Fase 3: Feature Engineering

In [ ]:
# Criação da nova variável numérica sugerida no projeto:
# potencia_w = velocidade_rotacao_rpm * torque_nm
df_tratado["potencia_w"] = (
    df_tratado["velocidade_rotacao_rpm"] * df_tratado["torque_nm"]
)

print("Valores nulos em potencia_w:", df_tratado["potencia_w"].isna().sum())
display(df_tratado[["velocidade_rotacao_rpm", "torque_nm", "potencia_w"]].head())

### Justificativa da nova variável

A potência operacional aproximada (`potencia_w`) combina velocidade de rotação e torque. Em manutenção preditiva, essa variável ajuda a representar o esforço mecânico da máquina, podendo capturar situações em que alta rotação e alto torque aumentam a exigência do equipamento.

## Fase 4: Divisão e Balanceamento dos Dados

In [ ]:
TARGET = "falha_maquina"

# Colunas que NÃO podem entrar no modelo:
# - falha_maquina: variável alvo
# - falhas específicas: motivos históricos da quebra, removidos para evitar data leakage
# - udi e id_produto: identificadores sem valor físico preditivo direto
colunas_vazamento = ["falha_twf", "falha_hdf", "falha_pwf", "falha_osf", "falha_rnf"]
colunas_identificadoras = ["udi", "id_produto"]

colunas_remover = [TARGET] + colunas_vazamento + colunas_identificadoras

X = df_tratado.drop(columns=colunas_remover)
y = df_tratado[TARGET].astype(int)

# Transformação da variável categórica tipo em dummies
X = pd.get_dummies(X, columns=["tipo"], drop_first=False)

print("Colunas utilizadas em X:")
display(X.columns.tolist())

print("\nConfirmação de segurança:")
print("falha_maquina em X?", TARGET in X.columns)
print("colunas de falhas específicas em X?", any(col in X.columns for col in colunas_vazamento))

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Distribuição do alvo no treino antes do SMOTE:")
display(y_train.value_counts())

print("Distribuição do alvo no teste:")
display(y_test.value_counts())

In [ ]:
# Balanceamento somente no treino para evitar vazamento de dados
smote = SMOTE(random_state=RANDOM_STATE)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("Distribuição do alvo no treino após SMOTE:")
display(y_train_bal.value_counts())

### Justificativa do balanceamento

O SMOTE foi aplicado **exclusivamente no conjunto de treino**. O conjunto de teste foi mantido com a distribuição original para simular melhor o cenário real e evitar vazamento de dados.

## Fase 5: Escalonamento de Variáveis (StandardScaler)

In [ ]:
variaveis_continuas = [
    "temperatura_ar_k",
    "temperatura_processo_k",
    "velocidade_rotacao_rpm",
    "torque_nm",
    "desgaste_ferramenta_min",
    "potencia_w"
]

# Cópias específicas para o KNN
X_train_knn = X_train_bal.copy()
X_test_knn = X_test.copy()

scaler = StandardScaler()

# fit_transform apenas no treino; transform no teste
X_train_knn[variaveis_continuas] = scaler.fit_transform(X_train_knn[variaveis_continuas])
X_test_knn[variaveis_continuas] = scaler.transform(X_test_knn[variaveis_continuas])

# Para a árvore, os dados permanecem sem escalonamento
X_train_tree = X_train_bal.copy()
X_test_tree = X_test.copy()

print("Escalonamento aplicado apenas para o KNN.")

### Justificativa do escalonamento

O KNN calcula distância entre observações; portanto, variáveis em escalas maiores podem dominar o cálculo. Por isso, o `StandardScaler` foi aplicado apenas nele.

A Árvore de Decisão realiza divisões por limiares nas variáveis e não depende de distância entre pontos. Assim, ela é naturalmente pouco sensível à escala dos atributos.

## Fase 6: Ajuste de Parâmetros e Combate ao Overfitting

In [ ]:
resultados_knn = []

for k in [3, 5, 7, 9, 11]:
    modelo_knn = KNeighborsClassifier(n_neighbors=k)
    modelo_knn.fit(X_train_knn, y_train_bal)

    acc_treino = accuracy_score(y_train_bal, modelo_knn.predict(X_train_knn))
    acc_teste = accuracy_score(y_test, modelo_knn.predict(X_test_knn))

    resultados_knn.append({
        "modelo": "KNN",
        "parametro": f"k={k}",
        "acuracia_treino": acc_treino,
        "acuracia_teste": acc_teste,
        "gap_treino_teste": acc_treino - acc_teste
    })

resultados_knn_df = pd.DataFrame(resultados_knn)
display(resultados_knn_df.sort_values("acuracia_teste", ascending=False))

In [ ]:
resultados_arvore = []

for depth in [3, 5, 7, None]:
    modelo_arvore = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    modelo_arvore.fit(X_train_tree, y_train_bal)

    acc_treino = accuracy_score(y_train_bal, modelo_arvore.predict(X_train_tree))
    acc_teste = accuracy_score(y_test, modelo_arvore.predict(X_test_tree))

    resultados_arvore.append({
        "modelo": "Árvore de Decisão",
        "parametro": f"max_depth={depth}",
        "acuracia_treino": acc_treino,
        "acuracia_teste": acc_teste,
        "gap_treino_teste": acc_treino - acc_teste
    })

resultados_arvore_df = pd.DataFrame(resultados_arvore)
display(resultados_arvore_df.sort_values("acuracia_teste", ascending=False))

### Modelo 3: LightGBM

O LightGBM é um algoritmo de *gradient boosting* baseado em árvores. Ele é treinado com os mesmos dados balanceados por SMOTE usados pelos demais modelos. Como árvores não dependem de escala, são utilizadas as variáveis imputadas, sem padronização. São comparadas configurações de complexidade diferentes para observar desempenho e risco de overfitting.


In [ ]:
configuracoes_lightgbm = [
    {"n_estimators": 100, "learning_rate": 0.05, "num_leaves": 15},
    {"n_estimators": 200, "learning_rate": 0.05, "num_leaves": 31},
    {"n_estimators": 300, "learning_rate": 0.03, "num_leaves": 31},
]

resultados_lightgbm = []

for configuracao in configuracoes_lightgbm:
    modelo_lightgbm = LGBMClassifier(
        **configuracao,
        random_state=RANDOM_STATE,
        verbosity=-1,
        n_jobs=-1
    )
    modelo_lightgbm.fit(X_train_tree, y_train_bal)

    acc_treino = accuracy_score(y_train_bal, modelo_lightgbm.predict(X_train_tree))
    acc_teste = accuracy_score(y_test, modelo_lightgbm.predict(X_test_tree))
    parametro = (
        f"n_estimators={configuracao['n_estimators']}, "
        f"learning_rate={configuracao['learning_rate']}, "
        f"num_leaves={configuracao['num_leaves']}"
    )

    resultados_lightgbm.append({
        "modelo": "LightGBM",
        "parametro": parametro,
        "acuracia_treino": acc_treino,
        "acuracia_teste": acc_teste,
        "gap_treino_teste": acc_treino - acc_teste
    })

resultados_lightgbm_df = pd.DataFrame(resultados_lightgbm)
display(resultados_lightgbm_df.sort_values("acuracia_teste", ascending=False))


In [ ]:
resultados_gerais = pd.concat([resultados_knn_df, resultados_arvore_df, resultados_lightgbm_df], ignore_index=True)
display(resultados_gerais.sort_values("acuracia_teste", ascending=False))

### Análise de overfitting

O overfitting é identificado quando a acurácia de treino fica muito superior à acurácia de teste. No KNN, valores menores de `k` tendem a memorizar mais os dados de treino. Na Árvore de Decisão, a configuração `max_depth=None` pode gerar maior risco de sobreajuste, pois permite crescimento livre da árvore.

A escolha final deve considerar a maior acurácia no teste e o menor gap possível entre treino e teste, priorizando estabilidade e generalização.

## Fase 7: Avaliação da Acurácia e Veredito Final

In [ ]:
melhor_knn_info = resultados_knn_df.sort_values("acuracia_teste", ascending=False).iloc[0]
melhor_arvore_info = resultados_arvore_df.sort_values("acuracia_teste", ascending=False).iloc[0]
melhor_lightgbm_info = resultados_lightgbm_df.sort_values("acuracia_teste", ascending=False).iloc[0]

print("Melhor KNN:")
display(melhor_knn_info)

print("Melhor Árvore de Decisão:")
display(melhor_arvore_info)

print("Melhor LightGBM:")
display(melhor_lightgbm_info)


In [ ]:
# Retreinamento dos melhores modelos para relatório final
melhor_k = int(melhor_knn_info["parametro"].split("=")[1])
melhor_depth_texto = melhor_arvore_info["parametro"].split("=")[1]
melhor_depth = None if melhor_depth_texto == "None" else int(melhor_depth_texto)
indice_melhor_lightgbm = resultados_lightgbm_df["acuracia_teste"].idxmax()
melhor_config_lightgbm = configuracoes_lightgbm[indice_melhor_lightgbm]

melhor_knn = KNeighborsClassifier(n_neighbors=melhor_k)
melhor_knn.fit(X_train_knn, y_train_bal)
pred_knn = melhor_knn.predict(X_test_knn)

melhor_arvore = DecisionTreeClassifier(max_depth=melhor_depth, random_state=RANDOM_STATE)
melhor_arvore.fit(X_train_tree, y_train_bal)
pred_arvore = melhor_arvore.predict(X_test_tree)

melhor_lightgbm = LGBMClassifier(
    **melhor_config_lightgbm,
    random_state=RANDOM_STATE,
    verbosity=-1,
    n_jobs=-1
)
melhor_lightgbm.fit(X_train_tree, y_train_bal)
pred_lightgbm = melhor_lightgbm.predict(X_test_tree)

acc_final_knn = accuracy_score(y_test, pred_knn)
acc_final_arvore = accuracy_score(y_test, pred_arvore)
acc_final_lightgbm = accuracy_score(y_test, pred_lightgbm)

print(f"Acurácia final do melhor KNN: {acc_final_knn:.4f}")
print(f"Acurácia final da melhor Árvore de Decisão: {acc_final_arvore:.4f}")
print(f"Acurácia final do melhor LightGBM: {acc_final_lightgbm:.4f}")


In [ ]:
print("Matriz de confusão - Melhor KNN")
display(pd.DataFrame(confusion_matrix(y_test, pred_knn),
                     index=["Real 0", "Real 1"],
                     columns=["Previsto 0", "Previsto 1"]))

print("\nRelatório de classificação - Melhor KNN")
print(classification_report(y_test, pred_knn))

In [ ]:
print("Matriz de confusão - Melhor Árvore de Decisão")
display(pd.DataFrame(confusion_matrix(y_test, pred_arvore),
                     index=["Real 0", "Real 1"],
                     columns=["Previsto 0", "Previsto 1"]))

print("\nRelatório de classificação - Melhor Árvore de Decisão")
print(classification_report(y_test, pred_arvore))

In [ ]:
print("Matriz de confusão - Melhor LightGBM")
display(pd.DataFrame(confusion_matrix(y_test, pred_lightgbm),
                     index=["Real 0", "Real 1"],
                     columns=["Previsto 0", "Previsto 1"]))

print("\nRelatório de classificação - Melhor LightGBM")
print(classification_report(y_test, pred_lightgbm))


In [ ]:
acuracias_finais = {
    "KNN": acc_final_knn,
    "Árvore de Decisão": acc_final_arvore,
    "LightGBM": acc_final_lightgbm,
}
modelo_final = max(acuracias_finais, key=acuracias_finais.get)
acuracia_final = acuracias_finais[modelo_final]

print(f"Modelo recomendado: {modelo_final}")
print(f"Acurácia no teste: {acuracia_final:.4f}")


### Veredito final

O modelo recomendado é aquele que apresentou maior acurácia na base de teste, mantendo a análise de overfitting sob controle. Para uso empresarial, também seria recomendável acompanhar métricas adicionais como recall da classe 1, pois em manutenção preditiva deixar de detectar uma falha pode gerar custo operacional elevado.

## 8. Exportação opcional da base limpa

In [ ]:
OUTPUT_PATH = Path("data") / "manutencao_preditiva_limpa.csv"
df_tratado.to_csv(OUTPUT_PATH, index=False)
print(f"Base limpa salva em: {OUTPUT_PATH}")